In [1]:
#import basic library
import pandas as pd

In [2]:
#load the data
train = pd.read_csv(r"../data/fraudTrain.csv",index_col=0)
test = pd.read_csv(r"../data/fraudTest.csv",index_col=0)

Model Training

In [3]:
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from imblearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report

In [4]:
#preprocess train and test datasets
import sys
import os

sys.path.append(os.path.abspath(".."))
from backend.app.services.feature_engineering import engineer_features,extract_features

#train
train = engineer_features(train)
train = extract_features(train)

#test

test = engineer_features(test)
test = extract_features(test)

In [5]:
#Divide the data into input and output features
X_train = train.drop("is_fraud",axis=1)
y_train = train.is_fraud

X_test = test.drop("is_fraud",axis=1)
y_test = test.is_fraud

In [6]:
#Divide numerical and categorical columns
numeric_cols = X_train.select_dtypes(include="number").columns
cat_cols = X_train.select_dtypes(include="object").columns

In [7]:
#Preprocessing pipeline
preprocessor = ColumnTransformer([
    ("scaler",StandardScaler(),numeric_cols),
    ("encoder",OneHotEncoder(handle_unknown="ignore"),cat_cols)
])

In [11]:
#RF Model pipeline
rf_pipe = Pipeline([
    ("preprocessor",preprocessor),
    ("smote",SMOTE()),
    ("model",RandomForestClassifier(n_estimators=100,max_depth=6,random_state=42,n_jobs=-1))
])



In [12]:
#XGB Model pipeline
xgb_pipe = Pipeline([
    ("preprocessor",preprocessor),
    ("smote",SMOTE()),
    ("model",XGBClassifier(n_estimators=300,max_depth=6,random_state=42,n_jobs=-1,tree_method="hist"))
])



In [13]:
models = {
    "rf":rf_pipe,
    "xgb":xgb_pipe
}

trained_models = {}

for name,model in models.items():
    print(f"{name} training started...")
    model.fit(X_train,y_train)
    print(f"{name} training finished...")

    

rf training started...
rf training finished...
xgb training started...
xgb training finished...


### RF Model Evaluation

In [14]:
#RF model Evaluation
y_pred = rf_pipe.predict(X_test)
print("Training eff: ",rf_pipe.score(X_train,y_train))
print("Testing eff: ",rf_pipe.score(X_test,y_test))
print(classification_report(y_test,y_pred))

Training eff:  0.977792430639906
Testing eff:  0.9784171496745657
              precision    recall  f1-score   support

           0       1.00      0.98      0.99    553574
           1       0.13      0.77      0.22      2145

    accuracy                           0.98    555719
   macro avg       0.56      0.87      0.60    555719
weighted avg       1.00      0.98      0.99    555719



Accuracy is good but precision is too low for class 1

### XGB Model Evaluation

In [15]:
#Prediction
y_pred = xgb_pipe.predict(X_test)

In [16]:
print("Training eff: ",xgb_pipe.score(X_train,y_train))
print("Testing eff: ",xgb_pipe.score(X_test,y_test))

Training eff:  0.9990244278635741
Testing eff:  0.9985586240528037


In [18]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred)

array([[553165,    409],
       [   392,   1753]], dtype=int64)

In [19]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    553574
           1       0.81      0.82      0.81      2145

    accuracy                           1.00    555719
   macro avg       0.91      0.91      0.91    555719
weighted avg       1.00      1.00      1.00    555719



In [20]:
#cheking model stability with cross validation
from sklearn.model_selection import cross_val_score
scores  = cross_val_score(xgb_pipe, X_train, y_train, cv=5, scoring="f1")
print(scores)

[0.84322034 0.84665974 0.86098655 0.8302926  0.84901828]


cv scores are stable so the pipeline is not overfitting